# Extract Writing Elements from Chess PDFs (Tesseract OCR)

Extract individual character glyphs (letters, digits, punctuation) from the chess notation PDFs,
using **Tesseract OCR** for character-level bounding boxes.

**Why Tesseract?**
- Gives exact per-character bounding boxes (not word estimates)
- Fast on printed text
- Simple API: `pytesseract.image_to_data()` → precise crops
- No variable-width character estimation errors

**Approach:**
1. Convert each PDF page to an image
2. Run Tesseract OCR → get character boxes + text
3. For each character: crop the exact region from the page image
4. Filter & organize: lowercase, uppercase, digit, punctuation, space
5. Zip everything for training

**Output:** `writing_elements_classifier.zip` with clean, precise glyphs.

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

## Step 2 — Install dependencies

In [ ]:
!pip install -q pytesseract pdf2image pillow numpy pandas
!apt-get install -qq tesseract-ocr poppler-utils  # Tesseract + PDF tools

## Step 3 — Configuration

In [ ]:
import os
import string
import zipfile
import shutil
from collections import defaultdict
from pathlib import Path

import numpy as np
import pytesseract
import pandas as pd
from pdf2image import convert_from_path
from PIL import Image

# ── EDIT THESE: paths to the PDFs on your Drive ────────────────────────
PDF_PATHS_ON_DRIVE = [
    '/content/gdrive/MyDrive/entrainement_ocr_echecs/pdfs/notation_figurine/CommentDevenirSuperAttaquant.pdf',
    '/content/gdrive/MyDrive/entrainement_ocr_echecs/pdfs/notation_figurine/ChessTrategy_Grivas_1.pdf',
    '/content/gdrive/MyDrive/entrainement_ocr_echecs/pdfs/notation_figurine/BoussoleSurEchiquier.pdf',
]

OUTPUT_DIR = '/content/writing_elements_glyphs'
PADDING = 2  # pixels of padding around character bbox (Tesseract is already precise)
MIN_GLYPH_SIZE = 6  # minimum width/height in pixels
MAX_GLYPH_SIZE = 128  # maximum width/height in pixels

# Character filter: what to extract
VALID_CHARS = set(string.ascii_letters + string.digits + '.,!?;:\'"()[]{}/-—–')

print(f'Valid characters to extract: {sorted(VALID_CHARS)}')
print(f'Output directory: {OUTPUT_DIR}')
print(f'Padding: {PADDING}px, Size range: {MIN_GLYPH_SIZE}—{MAX_GLYPH_SIZE}px')

# Check PDFs exist
for pdf_path in PDF_PATHS_ON_DRIVE:
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f'PDF not found: {pdf_path}')
print(f'\n✅ All {len(PDF_PATHS_ON_DRIVE)} PDFs found')

## Step 4 — Extract characters from PDFs

In [ ]:
# Create output directory
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR)

# Organize by character type
def get_char_category(c):
    """Return a category for organizing glyphs."""
    if c == ' ':
        return 'space'
    elif c in string.ascii_lowercase:
        return 'lowercase'
    elif c in string.ascii_uppercase:
        return 'uppercase'
    elif c in string.digits:
        return 'digit'
    else:
        return 'punctuation'

# Create subdirectories
for cat in ['lowercase', 'uppercase', 'digit', 'punctuation', 'space']:
    os.makedirs(os.path.join(OUTPUT_DIR, cat), exist_ok=True)

# Track statistics
stats = defaultdict(int)
char_counts = defaultdict(int)
glyph_id = 0

print(f'\n{"="*70}')
print('EXTRACTING CHARACTER GLYPHS WITH TESSERACT OCR')
print(f'{"="*70}')

for pdf_idx, pdf_path in enumerate(PDF_PATHS_ON_DRIVE):
    print(f'\n[{pdf_idx + 1}/{len(PDF_PATHS_ON_DRIVE)}] {os.path.basename(pdf_path)}')
    
    try:
        # Convert PDF to images
        print(f'  Converting pages to images...')
        page_images = convert_from_path(pdf_path, dpi=150)
        n_pages = len(page_images)
        print(f'  Pages: {n_pages}')
        
        for page_idx, page_image in enumerate(page_images):
            page_image_arr = np.array(page_image)
            page_h, page_w = page_image_arr.shape[:2]
            
            # Run Tesseract OCR to get character-level data
            ocr_data = pytesseract.image_to_data(page_image, output_type=pytesseract.Output.DATAFRAME)
            
            # Filter for actual character detections (conf > 0 means Tesseract found something)
            ocr_data = ocr_data[ocr_data['conf'] > 0]
            
            if len(ocr_data) == 0:
                stats['empty_pages'] += 1
                continue
            
            # Extract individual characters
            for idx, row in ocr_data.iterrows():
                char = row['text'].strip()
                
                # Skip empty or unwanted characters
                if not char or char not in VALID_CHARS:
                    stats['skipped_chars'] += 1
                    continue
                
                # Get bounding box from Tesseract (in pixels)
                x0 = int(row['left'])
                y0 = int(row['top'])
                w = int(row['width'])
                h = int(row['height'])
                x1 = x0 + w
                y1 = y0 + h
                
                # Add small padding
                x0_padded = max(0, x0 - PADDING)
                y0_padded = max(0, y0 - PADDING)
                x1_padded = min(page_w, x1 + PADDING)
                y1_padded = min(page_h, y1 + PADDING)
                
                crop_w = x1_padded - x0_padded
                crop_h = y1_padded - y0_padded
                
                # Filter by size
                if crop_w < MIN_GLYPH_SIZE or crop_h < MIN_GLYPH_SIZE:
                    stats['too_small'] += 1
                    continue
                if crop_w > MAX_GLYPH_SIZE or crop_h > MAX_GLYPH_SIZE:
                    stats['too_large'] += 1
                    continue
                
                try:
                    # Crop from page image
                    glyph_img = page_image_arr[y0_padded:y1_padded, x0_padded:x1_padded]
                    
                    # Convert to PIL, then grayscale
                    glyph_pil = Image.fromarray(glyph_img).convert('L')
                    
                    # Save
                    cat = get_char_category(char)
                    out_path = os.path.join(OUTPUT_DIR, cat, f'glyph_{glyph_id:08d}.png')
                    glyph_pil.save(out_path)
                    
                    char_counts[char] += 1
                    stats['saved'] += 1
                    glyph_id += 1
                except Exception as e:
                    stats['error'] += 1
                    continue
            
            if (page_idx + 1) % max(1, n_pages // 10) == 0:
                pct = 100 * (page_idx + 1) // n_pages
                print(f'    Pages: {pct:3d}%  (glyphs saved: {stats["saved"]})', flush=True)
    
    except Exception as e:
        print(f'  ❌ Error processing PDF: {e}')
        stats['pdf_error'] += 1

print(f'\n{"="*70}')
print('EXTRACTION COMPLETE')
print(f'{"="*70}')
print(f'\nStats:')
print(f'  Saved: {stats["saved"]}')
print(f'  Skipped (invalid char): {stats["skipped_chars"]}')
print(f'  Too small: {stats["too_small"]}')
print(f'  Too large: {stats["too_large"]}')
print(f'  Empty pages: {stats["empty_pages"]}')
print(f'  Errors: {stats["error"]}')
print(f'  PDF errors: {stats["pdf_error"]}')

print(f'\nTop 25 characters extracted:')
for char, count in sorted(char_counts.items(), key=lambda x: -x[1])[:25]:
    char_display = repr(char) if char != ' ' else "'SPACE'"
    print(f'  {char_display:10s}: {count:6d}')

## Step 5 — Organize into glyphs/ structure and verify

In [ ]:
# The structure is already organized, but let's rename to match the training pipeline
# Move everything into a 'glyphs' subfolder to match chess_glyphs_classifier.zip structure

FINAL_DIR = '/content/writing_elements_structure'
GLYPHS_DIR = os.path.join(FINAL_DIR, 'glyphs')

if os.path.exists(FINAL_DIR):
    shutil.rmtree(FINAL_DIR)
os.makedirs(GLYPHS_DIR, exist_ok=True)

# Move categories
for cat in os.listdir(OUTPUT_DIR):
    src = os.path.join(OUTPUT_DIR, cat)
    dst = os.path.join(GLYPHS_DIR, cat)
    if os.path.isdir(src):
        shutil.move(src, dst)

print('Organized structure:')
for cat in sorted(os.listdir(GLYPHS_DIR)):
    cat_dir = os.path.join(GLYPHS_DIR, cat)
    n_imgs = len([f for f in os.listdir(cat_dir) if f.endswith('.png')])
    print(f'  {cat:15s}: {n_imgs:6d} glyphs')

total = sum(
    len([f for f in os.listdir(os.path.join(GLYPHS_DIR, cat)) if f.endswith('.png')])
    for cat in os.listdir(GLYPHS_DIR)
)
print(f'\n✅ Total: {total} glyphs')

## Step 6 — Sample a few glyphs (sanity check)

In [ ]:
import matplotlib.pyplot as plt
import random

# Show a few random samples from each category
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()

for ax_idx, cat in enumerate(sorted(os.listdir(GLYPHS_DIR))):
    if ax_idx >= len(axes):
        break
    
    cat_dir = os.path.join(GLYPHS_DIR, cat)
    imgs = [f for f in os.listdir(cat_dir) if f.endswith('.png')]
    
    if not imgs:
        axes[ax_idx].text(0.5, 0.5, f'{cat}\n(no glyphs)', ha='center', va='center')
        axes[ax_idx].set_title(cat)
        continue
    
    # Load a random sample
    sample_file = random.choice(imgs)
    sample_path = os.path.join(cat_dir, sample_file)
    img = Image.open(sample_path).convert('L')
    
    axes[ax_idx].imshow(img, cmap='gray')
    axes[ax_idx].set_title(f'{cat}\n({len(imgs)} glyphs)')
    axes[ax_idx].axis('off')

# Hide unused subplots
for ax_idx in range(len(os.listdir(GLYPHS_DIR)), len(axes)):
    axes[ax_idx].axis('off')

plt.tight_layout()
plt.show()

print('✅ Samples look good!')

## Step 7 — Create zip file

In [ ]:
ZIP_PATH = '/content/writing_elements_classifier.zip'

print(f'Creating zip: {ZIP_PATH}')

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(FINAL_DIR):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, FINAL_DIR)
            zf.write(file_path, arcname)
            if len(files) % 1000 == 0:
                print(f'  {len(files)} files processed...', flush=True)

zip_size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
print(f'\n✅ Created {ZIP_PATH} ({zip_size_mb:.1f} MB)')

## Step 8 — Upload zip to Drive

In [ ]:
import shutil

DRIVE_UPLOAD_PATH = '/content/gdrive/MyDrive/entrainement_ocr_echecs/writing_elements_classifier.zip'

print(f'Uploading to Drive: {DRIVE_UPLOAD_PATH}')
shutil.copy2(ZIP_PATH, DRIVE_UPLOAD_PATH)
print(f'✅ Uploaded!')

## Step 9 — Download zip locally (optional)

In [ ]:
from google.colab import files
files.download(ZIP_PATH)
print(f'✅ Downloaded {ZIP_PATH}')